# Enfoque híbrido

**Fase 1 (Keywords):** En esta fase se realiza la asignación de pertinencias en todos los proyectos con coincidencias de keywords registradas en el diccionario. Esta fase es exactamente igual al enfoque original de matching directo, contando solo con la adición de un registro de asignaciones como salida donde se muestra que palabras hicieron el matching en cada proyecto con 1.

**Fase 2 (Embeddings):** Es opcional y se ejecuta solo si el usuario lo indica y en el indicador que se especifique. Esta tiene el objetivo de encontrar similitudes entre los textos de los proyectos aún con 0 y las keywords registradas y mostrarlas al usuario/desarrollador con el objetivo de que este las analice y expanda el diccionario según lo considere. Esta fase como tal no realiza asignaciones, ya que se considera que, en caso contrario, se pierde el control de los criterios de asignación al depender del entrenamiento del modelo, por lo que se mantiene como apoyo para la ubicación de posibles falsos negativos y expansión del diccionario a largo plazo.


## Imports y variables globales

In [2]:
import json
import re
import numpy as np
import pandas as pd
import unicodedata

In [3]:
# Bandera para definir si el modo debug está activo
DEBUG = True

# Umbral para "cosechar" los proyectos al menos un 40% similitud con el conjunto de keywords
PRIMER_UMBRAL = 0.4

# Umbral para mostrar en terminal los fragmentos del proyecto con similitud de al menos un 50% con la keyword más similar.
SEGUNDO_UMBRAL = 0.4

## Funciones de utilidad

### Normalizar texto

In [4]:
# Función para normalizar el texto, elimina tildes, y convierte el texto a minúsculas
def normalizar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    return texto

### Cargar modelo

In [5]:
_modelo = None

# Función para cargar modelo solo si se indica como argumento
def cargar_modelo():
    global _modelo, util
    if _modelo is None:
        print("Cargando modelo de embeddings...")
        from sentence_transformers import SentenceTransformer, util
        # Modelo preentrenado
        _modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    return _modelo

### Construir texto

In [6]:
# Función para concatenar las columnas cualitativas de un proyecto en un solo texto
def construir_texto(row, columnas_texto, columnas_disponibles):
    partes = []
    for col in columnas_texto:
        if col in columnas_disponibles:
            valor = row.get(col, "")
            if isinstance(valor, str) and valor.strip():
                partes.append(f"{col}: {valor.strip()}")
    return "\n".join(partes)

### Mostrar conteos

In [7]:
# Función para mostrar los conteos
def mostrar_conteos(base, descripciones, etiqueta):
    print(f"\nConteos - {etiqueta}:")
    resultados = base[list(descripciones.keys())].sum().astype(int)
    for indicador, total in resultados.items():
        print(f"  {indicador:<70} {total}")

## Fase 1: Keywords

Busca palabras clave exactas en el texto de cada proyecto. Marca 1 donde hay coincidencias.

In [8]:
# Función para ejecutar la fase 1 de keywords
def fase_keywords(base, textos_proyectos, descripciones):
    matches_por_indicador = {ind: set() for ind in descripciones}

    print("\n--- Fase 1: Keywords ---")

    for indicador, info in descripciones.items():
        keywords = info.get("palabras_clave", [])
        if not keywords:
            continue

        if DEBUG:
            print(f"\n=== KEYWORDS {indicador} ===")

        for idx, texto in enumerate(textos_proyectos):
            texto_lower = normalizar(texto)
            coincidencias = []

            for palabra in keywords:
                patron = r"\b" + re.escape(normalizar(palabra)) + r"\b"
                if re.search(patron, texto_lower):
                    coincidencias.append(palabra)

            if coincidencias:
                matches_por_indicador[indicador].add(idx)
                base.loc[idx, indicador] = 1

                if DEBUG:
                    print(f"  [KW] Proyecto {idx + 2}: {coincidencias}")

    return matches_por_indicador

## Fase 2: Embeddings

Sugiere proyectos candidatos para enriquecer el diccionario de keywords. NO asigna valores, solo imprime en consola para revisión.

El objetivo de esta fase es dar opciones al usuario/desarrollador de posibles keywords o frases similares utilizando el JSON de descripciones_indicadores, esto con el objetivo del enriquecimiento del diccionario a la vez de evitar asignaciones abstractas de parte del modelo.

In [9]:
# Función para preparar el modelo y los embeddings de los proyectos (una sola vez)
def preparar_embeddings(textos_proyectos):
    modelo = cargar_modelo()
    total_proyectos = len(textos_proyectos)
    print(f"\nCalculando embeddings de {total_proyectos} proyectos...")
    embeddings_proyectos = modelo.encode(
        textos_proyectos, show_progress_bar=True, batch_size=64, convert_to_numpy=True,
    )
    return modelo, embeddings_proyectos


# Función para evaluar un solo indicador contra los proyectos sin match de keywords
def evaluar_indicador(indicador, info, textos_proyectos, embeddings_proyectos, matches_keywords,
                       modelo, primer_umbral=None, segundo_umbral=None):
    primer_umbral = PRIMER_UMBRAL if primer_umbral is None else primer_umbral
    segundo_umbral = SEGUNDO_UMBRAL if segundo_umbral is None else segundo_umbral

    total_proyectos = len(textos_proyectos)
    indices_todos = set(range(total_proyectos))
    sin_match = sorted(indices_todos - matches_keywords.get(indicador, set()))

    if not sin_match:
        print(f"\n=== {indicador}: todos cubiertos por keywords, skip ===")
        return

    keywords = info.get("palabras_clave", [])
    texto_indicador = " | ".join(keywords) if keywords else info.get("descripcion", indicador)
    embedding_indicador = modelo.encode(texto_indicador, convert_to_numpy=True)

    embeddings_sub = embeddings_proyectos[sin_match]
    similitudes = util.cos_sim(embedding_indicador, embeddings_sub)[0]

    print(f"\n=== EMBEDDING {indicador} (evaluando {len(sin_match)} proyectos, "
          f"umbral1={primer_umbral}, umbral2={segundo_umbral}) ===")

    encontrados = 0
    for i, idx_proyecto in enumerate(sin_match):
        score = float(similitudes[i])
        if score < primer_umbral:
            continue

        oraciones = [s.strip() for s in re.split(r'[.\n|]', textos_proyectos[idx_proyecto]) if s.strip()]
        if oraciones:
            emb_oraciones = modelo.encode(oraciones, convert_to_numpy=True)
            sims_oraciones = util.cos_sim(embedding_indicador, emb_oraciones)[0]
            mejor_idx = int(sims_oraciones.argmax())
            mejor_frag = oraciones[mejor_idx][:250]
        else:
            mejor_frag = "(sin fragmento)"

        if keywords:
            emb_kws = modelo.encode(keywords, convert_to_numpy=True)
            emb_proyecto = embeddings_proyectos[idx_proyecto]
            sims_kws = util.cos_sim(emb_proyecto, emb_kws)[0]
            mejor_kw_idx = int(sims_kws.argmax())
            mejor_kw = keywords[mejor_kw_idx]
            mejor_kw_score = float(sims_kws[mejor_kw_idx])
        else:
            mejor_kw, mejor_kw_score = "(sin keywords)", 0.0

        if mejor_kw_score > segundo_umbral:
            encontrados += 1
            print(
                f"  [EMB] Proyecto {idx_proyecto + 2}: score={score:.3f}\n"
                f"        Fragmento : {mejor_frag}\n"
                f"        Similar a : '{mejor_kw}' ({mejor_kw_score:.3f})\n"
            )

    if encontrados == 0:
        print("  (sin candidatos que superen ambos umbrales)")

## Procesamiento

### Carga de datos

In [10]:
ruta_entrada = "../../info/Indicadores_Pertinencia_VAS_2026.xlsx"

base = pd.read_excel(ruta_entrada)

with open("columnas.json", encoding="utf-8") as f:
    columnas = json.load(f)

with open("descripciones_indicadores.json", encoding="utf-8") as f:
    descripciones = json.load(f)

descripciones.pop("inactivos", None)

for indicador in descripciones:
    base[indicador] = 0

columnas_disponibles = set(base.columns)
textos_proyectos = base.apply(
    construir_texto, axis=1,
    columnas_texto=columnas["columnas_texto"],
    columnas_disponibles=columnas_disponibles,
).tolist()

print(f"{len(textos_proyectos)} proyectos cargados.")

2972 proyectos cargados.


### Fase 1

In [11]:
matches_keywords = fase_keywords(base, textos_proyectos, descripciones)
mostrar_conteos(base, descripciones, "después de keywords")


--- Fase 1: Keywords ---

=== KEYWORDS A.1.1 Agua ===
  [KW] Proyecto 32: ['agua']
  [KW] Proyecto 57: ['mar']
  [KW] Proyecto 90: ['agua', 'rios']
  [KW] Proyecto 118: ['agua', 'acuatico']
  [KW] Proyecto 127: ['rio']
  [KW] Proyecto 130: ['agua']
  [KW] Proyecto 135: ['agua']
  [KW] Proyecto 143: ['rios']
  [KW] Proyecto 146: ['agua']
  [KW] Proyecto 159: ['rio', 'cuenca', 'cuencas']
  [KW] Proyecto 167: ['rios']
  [KW] Proyecto 207: ['agua']
  [KW] Proyecto 217: ['agua']
  [KW] Proyecto 219: ['agua', 'aguas', 'acueductos']
  [KW] Proyecto 229: ['rios', 'mar', 'mares', 'acuatica']
  [KW] Proyecto 245: ['agua', 'aguas', 'recurso hídrico']
  [KW] Proyecto 250: ['acueductos']
  [KW] Proyecto 289: ['agua']
  [KW] Proyecto 292: ['agua', 'aguas', 'recurso hídrico', 'potable']
  [KW] Proyecto 302: ['agua', 'recurso hídrico', 'potable', 'manantiales']
  [KW] Proyecto 354: ['agua']
  [KW] Proyecto 382: ['agua']
  [KW] Proyecto 391: ['agua']
  [KW] Proyecto 410: ['agua']
  [KW] Proyecto 417: 

### Preparar embeddings

In [12]:
modelo, embeddings_proyectos = preparar_embeddings(textos_proyectos)

Cargando modelo de embeddings...


/mnt/c/Users/Aron/Desktop/U/Cursos/2026/Asistencia/code/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 694.12it/s]



Calculando embeddings de 2972 proyectos...


Batches: 100%|██████████| 47/47 [01:50<00:00,  2.35s/it]


### Celda de trabajo

In [13]:
with open("descripciones_indicadores.json", encoding="utf-8") as f:
    descripciones_actualizado = json.load(f)
descripciones_actualizado.pop("inactivos", None)

indicador_actual = "A.1.2 Suelo"

evaluar_indicador(
    indicador_actual, descripciones_actualizado[indicador_actual],
    textos_proyectos, embeddings_proyectos, matches_keywords, modelo
)


=== EMBEDDING A.1.2 Suelo (evaluando 2554 proyectos, umbral1=0.4, umbral2=0.4) ===
  [EMB] Proyecto 136: score=0.472
        Fragmento : Descriptores: Botánica**Diversidad biológica**Educación ambiental**EDUCACIÓN AMBIENTAL**Enseñanza de la biología**Etnobotánica**ETNOBOTÁNICA**Jardin botánico**Taxonomía botánica
        Similar a : 'agrosilvicultura' (0.446)

  [EMB] Proyecto 149: score=0.407
        Fragmento : Descriptores: EDUCACIÓN**Educación masiva**EDUCACIÓN NUTRICIONAL**Eficiencia de la educación**NUTRICIÓN**Planificación de la educación**Proyecto de educación
        Similar a : 'agricultura' (0.439)

  [EMB] Proyecto 159: score=0.423
        Fragmento : Descriptores: COMUNIDAD**Cuenca**ESPACIO URBANO**GESTIÓN AMBIENTAL**Gestión de los recursos hídricos**GESTIÓN DE RECURSOS**Recursos territoriales**Relación escuela-comunidad
        Similar a : 'tierras' (0.420)

  [EMB] Proyecto 241: score=0.432
        Fragmento : Descriptores: ACTUALIZACIóN DE LOS CONOCIMIENTOS**Ingeniería

In [14]:
with open("descripciones_indicadores.json", encoding="utf-8") as f:
    descripciones = json.load(f)
descripciones.pop("inactivos", None)

indicador_a_mostrar = "A.1.2 Suelo"
base[indicador_a_mostrar] = 0

matches_keywords[indicador_a_mostrar] = fase_keywords(
    base, textos_proyectos, {indicador_a_mostrar: descripciones[indicador_a_mostrar]}
)[indicador_a_mostrar]

mostrar_conteos(base, {indicador_a_mostrar: descripciones[indicador_a_mostrar]}, "después de keywords")


--- Fase 1: Keywords ---

=== KEYWORDS A.1.2 Suelo ===
  [KW] Proyecto 13: ['tierra']
  [KW] Proyecto 20: ['tierra']
  [KW] Proyecto 31: ['tierra', 'mineral', 'minerales']
  [KW] Proyecto 32: ['suelo']
  [KW] Proyecto 39: ['agricultura', 'agricola']
  [KW] Proyecto 49: ['agricola']
  [KW] Proyecto 53: ['agricultura']
  [KW] Proyecto 81: ['agricultura', 'agroindustria']
  [KW] Proyecto 85: ['tierra']
  [KW] Proyecto 90: ['suelo', 'tierra', 'agronegocio', 'agroindustria']
  [KW] Proyecto 98: ['tierra']
  [KW] Proyecto 115: ['mineral']
  [KW] Proyecto 146: ['agricolas']
  [KW] Proyecto 170: ['agricultura', 'agricola', 'agricolas']
  [KW] Proyecto 172: ['agricultura', 'agricola', 'agricolas']
  [KW] Proyecto 188: ['agricultura', 'agricola']
  [KW] Proyecto 195: ['agricultura', 'agroindustria', 'agricolas']
  [KW] Proyecto 196: ['agroindustria']
  [KW] Proyecto 197: ['agricultura', 'agroindustria', 'agroindustrias', 'agroindustrial', 'agricola']
  [KW] Proyecto 199: ['suelo', 'agricultura'